# Importar bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Leitura da camada bronze

In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')

In [0]:
df.display()

# Retirando espaços em branco

In [0]:
for i in df.schema.fields:
    if isinstance(i.dataType, StringType):
        df = df.withColumn(i.name, F.trim(F.col(i.name)))

# Verificando valores distintos

In [0]:
df.select('GEN').distinct().display()

In [0]:
df.select('CID').distinct().display()

# Limpeza do campo CID

In [0]:
df = (
    df.withColumn(
        'CID', 
        F.when(F.col('CID').startswith('NAS'),
            F.substring(F.col('CID'), 4, F.length(F.col('CID')))
        ).otherwise(F.col('CID'))

    )
)

# Padronização do campo Gen

In [0]:
df.select('GEN').distinct().display()

In [0]:
df = (df.withColumn(
    'GEN',
    F.when(F.upper(F.col('GEN')).isin('M', 'MALE'), 'Male')
     .when(F.upper(F.col('GEN')).isin('F', 'FEMALE'), 'Female')
     .otherwise('n/a')
))#.select('GEN').distinct().display()


# Validação do campo de data

In [0]:
# display(
#     df.withColumn(
#         'BDATE',
#         F.when(F.col('BDATE') > F.current_date(), None)
#          .otherwise(F.col('BDATE'))
#     )
#     .select('BDATE')
#     .distinct()
#     .orderBy(F.col('BDATE').asc())
# )

df = (df.withColumn(
        'BDATE',
        F.when(F.col('BDATE') > F.current_date(), None)
         .otherwise(F.col('BDATE'))
    ))

# Renomeando colunas

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Escrita na tabela da camada prata

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_customers')

# Check criação da tabela

In [0]:
%sql
SELECT * 
FROM workspace.silver.erp_customers
LIMIT 10